# Regional VMD EOG removal — nonfrontal-review

nonfrontal-review

## Scientific question, inputs and contracts

Attach exact completed nonfrontal-full001/1 and matched-context-full001/1. Verify source/split identities and every consumed prediction. Hold nonfrontal outputs fixed and replace only frontal rows with archived calibrated raw regression; retain the matched VMD output. Report all seven arms, verified middle versus temporal subgroups, paper metrics, preservation, source-aware supplemental uncertainty and both channel-first and pooled-energy SNR. No new fit, neural model, AutoVMD, reserved data or native qualification.

## Fresh source and environment
Resolve latest published code once and pin it. New temporary clones avoid stale Python imports; original inputs remain immutable.

In [ ]:
import os, sys, json, tempfile, subprocess
from pathlib import Path
IN_KAGGLE = Path('/kaggle/input').is_dir()
if not IN_KAGGLE:
    raise RuntimeError('This numerical campaign runs on Kaggle only')
TEMP_ROOT = Path(tempfile.mkdtemp(prefix='vmd-eog-'))
CHECKOUT = TEMP_ROOT / 'repository'
REPOSITORY_URL = 'https://github.com/saltypal/Lateral-eye-Artifact-removal.git'
BRANCH = 'research/vmd-eog-removal-notebooks'
RUN_SPEC = {'stage': 'nonfrontal-review', 'run_id': 'nonfrontal-review-manual', 'git_sha': '', 'campaign_id': 'vmd-eog-20261010', 'profile': 'pilot'}
OUT = Path('/kaggle/working/results') / RUN_SPEC['campaign_id'] / RUN_SPEC['run_id']
OUT.mkdir(parents=True,exist_ok=False)
(OUT/'run_spec.json').write_text(json.dumps(RUN_SPEC,indent=2))
(OUT/'experiment.json').write_text(json.dumps(RUN_SPEC.get('experiment',{}),indent=2))
(OUT/'execution_state.json').write_text(json.dumps({'stage':RUN_SPEC['stage'],'status':'bootstrap_running'}))
print('BOOTSTRAP_START', RUN_SPEC['run_id'], RUN_SPEC['git_sha'], flush=True)
def bootstrap_call(command, phase):
    print('BOOTSTRAP_PHASE', RUN_SPEC['run_id'], phase, flush=True)
    result = subprocess.run(command)
    if result.returncode:
        (OUT/'execution_state.json').write_text(json.dumps({'stage':RUN_SPEC['stage'],
            'status':'failed','failure_layer':'bootstrap','phase':phase,'exit_code':result.returncode}))
        raise RuntimeError('Bootstrap failed during '+phase+'; retained run identity and status in '+str(OUT))
# Manual launches resolve latest once; CLI launches already have its exact SHA.
if not RUN_SPEC['git_sha']:
    line = subprocess.check_output(['git','ls-remote',REPOSITORY_URL,'refs/heads/'+BRANCH], text=True).strip()
    RUN_SPEC['git_sha'] = line.split()[0]
bootstrap_call(['git','clone','--quiet','--branch',BRANCH,'--single-branch',REPOSITORY_URL,str(CHECKOUT)], 'clone')
bootstrap_call(['git','-C',str(CHECKOUT),'checkout','--quiet',RUN_SPEC['git_sha']], 'checkout')
ACTUAL_SHA = subprocess.check_output(['git','-C',str(CHECKOUT),'rev-parse','HEAD'],text=True).strip()
if ACTUAL_SHA != RUN_SPEC['git_sha']:
    raise RuntimeError('Source SHA mismatch')
ENV_ROOT = TEMP_ROOT / 'environment'
bootstrap_call([sys.executable,'-m','venv','--without-pip','--system-site-packages',str(ENV_ROOT)], 'environment')
ENV_PY = ENV_ROOT / 'bin/python'
bootstrap_call([sys.executable,'-m','pip','--python',str(ENV_PY),'install','--quiet','--progress-bar','off','-r',str(CHECKOUT/'requirements.txt')], 'dependencies')
PROCESS_ENV = os.environ.copy()
PROCESS_ENV['PYTHONPATH'] = str(CHECKOUT/'src')
PROCESS_ENV['PYTHONUNBUFFERED'] = '1'
PROCESS_ENV['OMP_NUM_THREADS'] = '2'
(OUT/'run_spec.json').write_text(json.dumps(RUN_SPEC,indent=2))
(OUT/'experiment.json').write_text(json.dumps(RUN_SPEC.get('experiment',{}),indent=2))
print('Exact Git SHA:', ACTUAL_SHA, 'Stage:', RUN_SPEC['stage'], 'Output:',OUT,flush=True)


## Inspect the pinned algorithm contract
The source modules remain inspectable in this checkout. Mode vectors, references and paired targets have distinct roles; the accompanying explanation defines their shapes and inference boundaries.

In [ ]:
print((CHECKOUT/'docs/NONFRONTAL_CHALLENGERS_PLAN.md').read_text())
print('Frozen predictions only; no fitting or parameter selection.')
print('Channel-first and pooled-energy SNR are separate aggregations.')
print('Module: nonfrontal_review.py; two disjoint fields, one subtraction.')


## Execute reusable modules
HEOG/VEOG are teacher/evaluation information. Clean targets cannot enter inference. Failed numerical or scientific gates stop dependent work.

In [ ]:
command = [str(ENV_PY),'-m','vmd_eog.runner','--stage',RUN_SPEC['stage'],
           '--input','/kaggle/input','--output',str(OUT),'--profile',RUN_SPEC['profile'],
           '--config',RUN_SPEC.get('config_path','configs/campaign.json'),
           '--experiment',str(OUT/'experiment.json')]
process = subprocess.run(command,cwd=CHECKOUT,env=PROCESS_ENV)
if process.returncode:
    raise RuntimeError('Stage failed; inspect saved execution_state.json and kernel logs')
print((OUT/'execution_state.json').read_text())


## Saved configurations and result tables
These are immutable outputs from this run. Readiness, teacher eligibility and final scientific qualification are distinct decisions.

In [ ]:
import pandas as pd
from IPython.display import display
print((OUT/'frontend_review_summary.json').read_text())
display(pd.read_csv(OUT/'review_method_means.csv'))
display(pd.read_csv(OUT/'review_condition_means.csv'))
print((OUT/'review_gates.json').read_text())
display(pd.read_csv(OUT/'coverage_uncertainty_intervals.csv'))


## Inspect evidence and persist outputs
Native real EEG has no clean-reference SNR. Pass-through failures remain counted. Parameters and physiological frequency bands are distinct.

In [ ]:
from IPython.display import display, Image
for figure in sorted(OUT.glob('*.png')):
    display(Image(filename=str(figure)))
for report in sorted(OUT.glob('*summary.json')):
    print(report.name, report.read_text()[:16000])
print('Persist this completed kernel version and retrieve outputs before launching a dependent stage.')
